# Block 1: Basic Elements in Neural Networks

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/imatge-upc/aa2/blob/main/notebooks/aa2_1_3_losses.ipynb)

## 1.3: Loss Functions

This notebook introduces loss functions and their gradients using PyTorch:

*   **Regression Losses**: Comparing L1, MSE and Huber losses and their gradients.
*   **Classification Losses**: Computing cross-entropy from probabilities and logits.
*   **Learning Signals**: Comparing cross-entropy and MSE for confidently wrong predictions.
*   **Custom Losses**: Defining a contrastive loss and computing its gradients with `loss.backward()`.

We start with scalar predictions to see how each loss responds to an error, then explore classification probabilities and pairs of embeddings.

In [ ]:
import torch
from torch import nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

torch.manual_seed(0)

### Main built-in Loss Functions in PyTorch

| Loss | Typical use | Model output expected by the loss |
|---|---|---|
| [`nn.L1Loss`](https://pytorch.org/docs/stable/generated/torch.nn.L1Loss.html) | Regression; robust to outliers | Predictions |
| [`nn.MSELoss`](https://pytorch.org/docs/stable/generated/torch.nn.MSELoss.html) | Regression | Predictions |
| [`nn.HuberLoss`](https://pytorch.org/docs/stable/generated/torch.nn.HuberLoss.html) | Robust regression | Predictions |
| [`nn.BCEWithLogitsLoss`](https://pytorch.org/docs/stable/generated/torch.nn.BCEWithLogitsLoss.html) | Binary or multilabel classification | Logits |
| [`nn.CrossEntropyLoss`](https://pytorch.org/docs/stable/generated/torch.nn.CrossEntropyLoss.html) | Multiclass classification | Logits |
| [`nn.NLLLoss`](https://pytorch.org/docs/stable/generated/torch.nn.NLLLoss.html) | Multiclass classification; negative log-likelihood | Log-probabilities |
| [`nn.TripletMarginLoss`](https://pytorch.org/docs/stable/generated/torch.nn.TripletMarginLoss.html) | Metric learning | Embeddings |

The built-in loss modules above use `reduction="mean"` by default, so they normally return one scalar: the mean loss over the batch (B samples, if loss is computed on several samples). With `reduction="none"`, they return the individual losses.

### Module and Functional Interfaces

`torch.nn` contains more than network layers (and activations, nn.ReLU). It also provides losses as `nn.Module` objects. The equivalent functional operations are available in `torch.nn.functional`.

In [ ]:
prediction = torch.tensor([2.0, 4.0, 5.0])       # batch B=3
target = torch.tensor([1.0, 4.0, 7.0])

criterion = nn.L1Loss()                          # module interface
loss_module = criterion(prediction, target)

loss_functional = F.l1_loss(prediction, target)  # functional interface

print(f"Module:     {loss_module.item():.3f}")
print(f"Functional: {loss_functional.item():.3f}")

The module form is convenient when the loss has configuration such as `delta`, `margin`, `reduction` or class weights. The call arguments are unified, e.g. `criterion(prediction, target)`. A custom loss can nevertheless be an ordinary Python function: autograd records the tensor operations in exactly the same way.

### Regression: Compare L1, MSE and Huber

For a scalar prediction, define the residual

$$r = \hat{y} - y.$$

We can compare the losses by taking $y=0$ and varying the prediction $\hat{y}=r$. We use `reduction="none"` because we want one loss value for every residual.

In [ ]:
prediction = r = torch.linspace(-3, 3, 301)
y = zeros = torch.zeros_like(r)


l1 = nn.L1Loss(reduction="none")(prediction, y)
mse = nn.MSELoss(reduction="none")(prediction, y)
huber = nn.HuberLoss(delta=1.0, reduction="none")(prediction, y)

plt.figure(figsize=(7, 4))
plt.plot(r, l1, label="L1", linewidth=2)
plt.plot(r, mse, label="MSE (L2)", linewidth=2)
plt.plot(r, huber, label=r"Huber ($\delta=1$)", linewidth=1)
plt.xlabel(r"Residual $r=\hat{y}-y$")
plt.ylabel("Loss")
plt.ylim(0, 8)
plt.grid(alpha=0.3)
plt.legend()
plt.show()

**Observe:**

- MSE grows quadratically, so large errors have a strong influence.
- L1 grows linearly and is therefore less sensitive to outliers.
- Huber is quadratic near zero and linear for large residuals.

**Question:** What does each of these properties imply for the gradient used during learning?

### Loss Gradients: What Does the Optimizer See?

The loss value measures the error, but the **gradient** determines the update. Because $r=\hat{y}-y$, the gradient with respect to the prediction is

$$\frac{\partial L}{\partial \hat{y}} = \frac{dL}{dr}.$$

Here we plot the analytical derivatives: `sign(r)` for L1, `2 * r` for MSE, and `r.clamp(-delta, delta)` for Huber. For L1 at zero, we use zero, matching PyTorch's convention. During training, PyTorch computes these derivatives automatically with `loss.backward()`, as in notebook 1.2.

In [ ]:
delta = 1.0

gradients = {
    "L1": torch.sign(r),
    "MSE (L2)": 2 * r,
    "Huber": r.clamp(-delta, delta),
}

plt.figure(figsize=(7, 4))
for name, gradient in gradients.items():
    plt.plot(r, gradient, label=name, linewidth=1)

plt.axhline(0, color="black", linewidth=0.8)
plt.xlabel(r"Residual $r=\hat{y}-y$")
plt.ylabel(r"Gradient $\partial L / \partial \hat{y}$")
plt.grid(alpha=0.3)
plt.legend()
plt.show()

With MSE, large errors produce large gradients. With L1, all non-zero errors have the same gradient magnitude. Huber's gradient grows near zero and is capped at $\pm\delta$.

For L1, a small residual may reach or cross zero after one update, while a large residual can keep producing updates in the same direction over many steps. Equal gradient magnitudes do not mean equal effects over the whole training process. For the network parameters, the chain rule gives

$$\nabla_\theta L = \operatorname{sign}(r)\,\nabla_\theta \hat{y}.$$

Parameter gradients also depend on how the prediction changes with the parameters. L1 removes the residual magnitude from the loss derivative.

### Huber Loss: Change $\delta$

The parameter $\delta$ determines where the loss changes from quadratic to linear behaviour.

In [ ]:
plt.figure(figsize=(7, 4))

for delta in [0.5, 1.0, 2.0]:
    loss = nn.HuberLoss(delta=delta, reduction="none")(r, zeros)
    plt.plot(r, loss, label=fr"$\delta={delta}$", linewidth=1)

plt.xlabel(r"Residual $r=\hat{y}-y$")
plt.ylabel("Huber loss")
plt.ylim(0, 6)
plt.grid(alpha=0.3)
plt.legend()
plt.show()

> Change the values of `delta`. Which values make the curve more similar to L1 over the displayed range? Which values extend the quadratic region?

### Multiclass Classification: Compute Cross-Entropy

For $C$ classes, the model assigns a predicted probability $p_c$ to every class. Cross-entropy compares the target distribution $q$ with this predicted distribution:

$$H(q,p)=-\sum_{c=1}^{C} q_c \log p_c.$$

For an ordinary class label, $q$ is one-hot: it is 1 for the correct class and 0 for every other class. Therefore,

$$H(q,p)=-\log p_{\text{correct}}.$$

A confident correct prediction has a loss near zero. Assigning a very small probability to the correct class produces a large loss.

In [ ]:
# B=2, C=3 (two samples, three classes)
target = torch.tensor([0, 2])  # B=2, two samples
logits = torch.tensor([        # C=3, three classes
    [0.7, 0.8, -0.2],          # 1st sample, bad prediction  👎
    [0.1, 0.3,  2.5]           # 2nd sample, good prediction 👍
])

probabilities = F.softmax(logits, dim=1)
target_distribution = F.one_hot(target, num_classes=3).float()
print("Predicted probabilities:")
print(probabilities)
print("\nTarget distributions:")
print(target_distribution)

loss_per_example = -(target_distribution * torch.log(probabilities)).sum(dim=1)
loss_from_probabilities = loss_per_example.mean()

print("\nProbability assigned to each correct class:")
print(probabilities[torch.arange(len(target)), target])
print("\nCross-entropy for each example:")
print(loss_per_example)
print("\nMean cross-entropy:", loss_from_probabilities.item())

### From Probabilities to Logits

A network normally produces **logits**, not probabilities. Softmax converts the logits into probabilities. Mathematically, we could calculate

```python
probabilities = F.softmax(logits, dim=1)
loss = -torch.log(probability_of_correct_class).mean()
```

In actual training, computing `softmax` and then `log` separately is unnecessarily vulnerable to numerical problems. PyTorch therefore performs the combined operation directly from the logits:

In [ ]:
loss_ce = nn.CrossEntropyLoss()(logits, target)

print(f"Explicit probabilities: {loss_from_probabilities.item():.6f}")
print(f"CrossEntropyLoss:       {loss_ce.item():.6f}")

Thus, `CrossEntropyLoss` expects logits, not probabilities. Internally, it computes the stable equivalent of

```python
nn.NLLLoss()(F.log_softmax(logits, dim=1), target)
```

`NLLLoss` receives log-probabilities; `CrossEntropyLoss` receives logits. Both implementations compute the same cross-entropy.

### Binary Classification

`BCEWithLogitsLoss` combines a sigmoid operation with binary cross-entropy. Its targets are floating-point values with the same shape as the logits. It can be used for **binary classification**
and for **multilabel classification**.

In [ ]:
binary_logits = torch.tensor([1.2, -0.7, 0.1])  # B=3, C=2 (three samples, binary classification)
binary_target = torch.tensor([1.0, 0.0, 1.0])

loss = nn.BCEWithLogitsLoss()(binary_logits, binary_target)

print("Probabilities for inspection:", torch.sigmoid(binary_logits))
print("Loss:", loss.item())

### Classification Gradients: Compare Cross-Entropy and MSE

MSE can also compare predicted probabilities with classification targets. To see how its learning signal differs from cross-entropy, consider one binary example with target $y=1$. The model outputs a logit $z$, and the predicted probability is $p=\sigma(z)$.

| Loss | Value for $y=1$ | Gradient with respect to the logit $z$ |
|---|---|---|
| Binary cross-entropy (BCE) | $-\log p$ | $p-1$ |
| MSE on the probability | $(p-1)^2$ | $2(p-1)p(1-p)$ |


We differentiate with respect to the **logit**, through which the loss sends its learning signal back to the model. For MSE, the chain rule introduces the sigmoid derivative $p(1-p)$. For BCE, that factor cancels when differentiating the combined loss and sigmoid.

As in the regression comparison, we plot the analytical gradients. During training, `loss.backward()` computes them automatically.

In [ ]:
z = torch.linspace(-6, 6, 501)
p = torch.sigmoid(z)
target = torch.ones_like(z)  # The correct class is y = 1.

bce_values = nn.BCEWithLogitsLoss(reduction="none")(z, target)
mse_values = nn.MSELoss(reduction="none")(p, target)

bce_gradient = p - 1
mse_gradient = 2 * (p - 1) * p * (1 - p)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(z, bce_values, label="Binary cross-entropy", linewidth=1)
axes[0].plot(z, mse_values, label="MSE on probability", linewidth=1)
axes[0].set_ylabel("Loss")
axes[0].set_title("Loss for target y = 1")

axes[1].plot(z, bce_gradient, label="Binary cross-entropy", linewidth=1)
axes[1].plot(z, mse_gradient, label="MSE on probability", linewidth=1)
axes[1].axhline(0, color="black", linewidth=0.8)
axes[1].set_ylabel(r"Gradient $\partial L / \partial z$")
axes[1].set_title("Learning signal at the logit")

for ax in axes:
    ax.axvline(0, color="gray", linestyle=":", linewidth=1)
    ax.set_xlabel("Logit z (left: confidently wrong; right: confidently correct)")
    ax.grid(alpha=0.3)
    ax.legend()

plt.tight_layout()
plt.show()

**Observe:**

- **Confidently wrong** ($z\ll0$, $p\approx0$): BCE has a gradient near $-1$, while the MSE gradient approaches zero because the sigmoid saturates. MSE therefore gives a weak learning signal even though the prediction is very wrong.
- **Confidently correct** ($z\gg0$, $p\approx1$): both losses and both gradients approach zero.
- Both gradients are negative for this target: gradient descent pushes the logit upward, increasing the probability of the correct class.

Cross-entropy avoids this extra sigmoid saturation factor. MSE is still a valid loss on probabilities, but its learning signal differs.

>**Question:** How would the plots change if the target were $y=0$? Derive the expressions of the table and validate changing the code.

### Custom Loss: Compare Pairs of Embeddings

A custom loss is any differentiable computation that produces a scalar. The following contrastive loss encourages similar pairs to have nearby embeddings and different pairs to be separated by at least a margin.

Because the loss uses differentiable PyTorch operations, we can call `loss.backward()` and inspect the embedding gradients without deriving them by hand.

In [ ]:
def contrastive_loss(z1, z2, y, margin=1.0):
    """
    z1, z2: embeddings, shape (batch_size, embedding_dim)
    y: 1 for similar pairs; 0 for different pairs
    """
    d = torch.norm(z1 - z2, dim=1)

    positive_loss = y * d**2
    negative_loss = (1 - y) * torch.clamp(margin - d, min=0)**2

    return (positive_loss + negative_loss).mean()

In [ ]:
# B=3, D=2 (three pais, embedding dimension is 2)
z1 = torch.tensor([[0.0, 0.0], [0.0, 0.0], [0.0, 0.0]], requires_grad=True)
z2 = torch.tensor([[0.2, 0.1], [0.3, 0.0], [2.0, 0.0]], requires_grad=True)
y = torch.tensor([1.0, 0.0, 0.0])

loss = contrastive_loss(z1, z2, y, margin=1.0)
print("Pairwise distances:", torch.norm(z1 - z2, dim=1))
print("Mean contrastive loss:", loss.item())

loss.backward()
print("Gradients with respect to z1:", z1.grad, sep="\n")
print("Gradients with respect to z2:", z2.grad, sep="\n")

**Questions:** Interpret the three pairs separately.

1. The first pair is similar: what happens as its distance increases?
2. The second pair is different but lies inside the margin: does it contribute to the loss?
3. The third pair is different and lies outside the margin: does it contribute?

For reuse, this function could be packaged as an `nn.Module` that stores `margin`, just like PyTorch's built-in loss modules. This is optional: the ordinary function already works with autograd.

### Takeaways

- The choice of loss determines what the model is encouraged to learn.
- L1, MSE and Huber respond differently to large residuals.
- Check carefully whether a classification loss expects logits, log-probabilities or probabilities.
- Built-in losses are `nn.Module` objects, but any differentiable tensor computation can define a loss.